## Generate Embeddings

In [2]:
import torch
import numpy as np
import csv
import os
from tqdm import tqdm
import sys
sys.path.append("/DATA5/ashishu23/SURGE/fairseq")
from examples.shubert.models.shubert import SHubertModel, SHubertConfig

In [3]:
def load_model(checkpoint_path):
    cfg = SHubertConfig()
    model = SHubertModel(cfg)
    
    checkpoint = torch.load(checkpoint_path)
    
    state_dict = checkpoint['model'] if 'model' in checkpoint else checkpoint
    model.load_state_dict(state_dict, strict=False)
    
    model.eval()
    model.cuda()
    return model


In [4]:
def process_sample(model, face_path, left_hand_path, right_hand_path, body_posture_path):
    face = torch.from_numpy(np.load(face_path)).float().cuda()
    left_hand = torch.from_numpy(np.load(left_hand_path)).float().cuda()
    right_hand = torch.from_numpy(np.load(right_hand_path)).float().cuda()
    body_posture = torch.from_numpy(np.load(body_posture_path)).float().cuda()

    length = face.shape[0]
    source = [{
        "face": face,
        "left_hand": left_hand,
        "right_hand": right_hand,
        "body_posture": body_posture,
        "label_face": torch.zeros((length, 1)).cuda(),
        "label_left_hand": torch.zeros((length, 1)).cuda(),
        "label_right_hand": torch.zeros((length, 1)).cuda(),
        "label_body_posture": torch.zeros((length, 1)).cuda()
    }]
    
    with torch.no_grad():
        result = model.extract_features(source, padding_mask=None, kmeans_labels=None, mask=False)

    layer_outputs = []
    for layer in result['layer_results']:
        layer_output = layer[-1].squeeze(1)  # [T, D]
        layer_outputs.append(layer_output.cpu().numpy())

    features = np.stack(layer_outputs, axis=0)  # [L, T, D]
    return features


In [5]:
def main(csv_list, checkpoint_path, output_dir, index):
    model = load_model(checkpoint_path)
    os.makedirs(output_dir, exist_ok=True)
    
    for row in csv_list:
        cues_list = row[0].split('\t')
        face_path, left_hand_path, right_hand_path, body_posture_path = cues_list[:4]

        output_filename = f"{os.path.basename(face_path).rsplit('.',1)[0].rsplit('_',1)[0]}.npy"
        output_path = os.path.join(output_dir, output_filename)
        
        if os.path.exists(output_path):
            print(f"Skipping {output_path} as it already exists")
            continue

        features = process_sample(model, face_path, left_hand_path, right_hand_path, body_posture_path)
        np.save(output_path, features)


In [6]:
index = 0 
csv_path = 'poses.csv' 
checkpoint_path = 'checkpoint_836_400000.pt'  
output_dir = 'output_embeddings'  
batch_size = 1000

os.makedirs(output_dir, exist_ok=True)

fixed_list = []
with open(csv_path, 'r') as csvfile:
    reader = csv.reader(csvfile)
    for row in reader:
        fixed_list.append(row)

video_batches = [fixed_list[i:i + batch_size] for i in range(0, len(fixed_list), batch_size)]
csv_list = video_batches[index]

main(csv_list, checkpoint_path, output_dir, index)


/home/ashishu23/miniconda3/envs/shubert_train_env/lib/python3.10/site-packages/torch/nn/utils/weight_norm.py:28: UserWarning: torch.nn.utils.weight_norm is deprecated in favor of torch.nn.utils.parametrizations.weight_norm.
  warnings.warn("torch.nn.utils.weight_norm is deprecated in favor of torch.nn.utils.parametrizations.weight_norm.")
